In [14]:
import rioxarray

MAIN_PATH = '/storage/group/cxs1024/default/mehdi/Fall_River_Lake_Dam_Break/topo/'
input_asc  = MAIN_PATH + 'Flood_plain.asc'
output_asc = MAIN_PATH + 'Flood_plain_UTM14.asc'

# Load
ds = rioxarray.open_rasterio(input_asc)

# Set source CRS if missing in the .asc file
if ds.rio.crs is None:
    ds.rio.write_crs("EPSG:4326", inplace=True)

# Import Resampling from rasterio (this fixes the error)
from rasterio.enums import Resampling

# Reproject to UTM Zone 14N (meters)
reproj = ds.rio.reproject(
    "EPSG:32614",                      # UTM 14N
    resolution=(30, 30),               # output cell size in meters – change if needed
    resampling=Resampling.bilinear     # or .cubic, .nearest, .average, etc.
)

# Save as ASCII grid
reproj.rio.to_raster(
    output_asc,
    driver="AAIGrid",
    nodata=-9999.0
)

print("Done! Reprojected file (in meters):", output_asc)

Done! Reprojected file (in meters): /storage/group/cxs1024/default/mehdi/Fall_River_Lake_Dam_Break/topo/Flood_plain_UTM14.asc


In [15]:
import pandas as pd
from pyproj import Transformer

# Paths
csv_path   = 'mesh/dam_break_finer_area.csv'
output_csv = 'mesh/dam_break_finer_area_UTM14.csv'

# Read the headerless CSV (lon, lat)
df = pd.read_csv(csv_path, header=None, names=['lon', 'lat'])

print("Original data (degrees):")
print(df.head())

# Convert from WGS84 (degrees) → UTM Zone 14N (meters)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32614", always_xy=True)
df['easting'], df['northing'] = transformer.transform(df['lon'].values, df['lat'].values)

# Optional: round to 2 decimals (cleaner file)
df['easting']  = df['easting'].round(2)
df['northing'] = df['northing'].round(2)

# Save new CSV (you can keep or drop the original lon/lat columns)
df[['easting', 'northing']].to_csv(output_csv, index=False, header=False)
# If you also want the original lon/lat in the output file, use this line instead:
# df.to_csv(output_csv, index=False)

print("\nConversion complete!")
print("New file with meters:", output_csv)
print("\nPreview (meters):")
print(df[['easting', 'northing']].head())

Original data (degrees):
         lon        lat
0 -96.086419  37.651457
1 -96.034544  37.666566
2 -95.807405  37.567350
3 -95.815966  37.488782
4 -95.938853  37.492812

Conversion complete!
New file with meters: mesh/dam_break_finer_area_UTM14.csv

Preview (meters):
     easting    northing
0  757038.97  4171138.72
1  761563.35  4172959.16
2  781977.86  4162605.95
3  781516.95  4153860.27
4  770634.56  4153946.60


In [16]:
import numpy as np
import os
import matplotlib.pyplot as plt

# %matplotlib inline
import rasterio
import pyproj
from rasterio.warp import calculate_default_transform, reproject, Resampling
import os
from rasterio.plot import show

# Allow inline jshtml animations
from matplotlib import rc
rc('animation', html='jshtml')
import anuga

In [35]:
import numpy as np
from scipy import ndimage
import pandas as pd

def auto_rectify_dem(input_file, output_file):
    header = {}
    with open(input_file, 'r') as f:
        for _ in range(6):
            line = f.readline().strip().split()
            header[line[0].lower()] = float(line[1])
        data = np.loadtxt(f)

    nodata = header.get('nodata_value', -9999)
    rows, cols = np.where(data != nodata)
    
    # Calculate tilt angle
    bl_idx, br_idx = np.argmin(rows + cols), np.argmax(cols - rows)
    y1, x1 = rows[bl_idx], cols[bl_idx]
    y2, x2 = rows[br_idx], cols[br_idx]
    angle_rad = np.arctan2(y2 - y1, x2 - x1)
    angle_deg = np.degrees(angle_rad)
    
    # Rotate Array
    data_nan = np.where(data == nodata, np.nan, data)
    rotated_data = ndimage.rotate(data_nan, angle_deg, reshape=True, order=1, cval=np.nan)
    
    # Calculate Crop (Translation)
    mask = ~np.isnan(rotated_data)
    coords = np.argwhere(mask)
    y_min_crop, x_min_crop = coords.min(axis=0)
    y_max_crop, x_max_crop = coords.max(axis=0)
    final_rect = rotated_data[y_min_crop:y_max_crop+1, x_min_crop:x_max_crop+1]

    # Save DEM... (omitted for brevity, same as your code)

    # SAVE TRANSFORM STATE
    state = {
        'angle_rad': angle_rad,
        'orig_xll': header['xllcorner'],
        'orig_yll': header['yllcorner'],
        'orig_nrows': int(header['nrows']),
        'cellsize': header['cellsize'],
        'x_crop_offset': x_min_crop,
        'y_crop_offset': y_min_crop,
        'center_orig_px': np.array([data.shape[1]/2, data.shape[0]/2]),
        'center_rot_px': np.array([rotated_data.shape[1]/2, rotated_data.shape[0]/2])
    }
    return state
    # Run the automation

MAIN_PATH = '/storage/group/cxs1024/default/mehdi/Fall_River_Lake_Dam_Break/topo/'
topography_file = MAIN_PATH + 'Flood_plain_UTM14.asc'
state = auto_rectify_dem(topography_file, 'mesh/rectified_local_dem.asc')

In [42]:
def transform_csv_coordinates(csv_input, csv_output, state):
    """
    Transforms UTM coordinates to local rectified DEM coordinates
    and saves as a raw CSV (no header).
    """
    # Load raw CSV (assuming it has no header based on your example)
    df = pd.read_csv(csv_input, names=['utm_e', 'utm_n'])
    
    # 1. Convert UTM to original Pixel Coordinates
    px_x = (df['utm_e'] - state['orig_xll']) / state['cellsize']
    px_y = state['orig_nrows'] - ((df['utm_n'] - state['orig_yll']) / state['cellsize'])
    
    # 2. Center-relative coordinates for rotation
    shifted_x = px_x - state['center_orig_px'][0]
    shifted_y = px_y - state['center_orig_px'][1]
    
    # 3. Apply Rotation Matrix
    cos_a = np.cos(state['angle_rad'])
    sin_a = np.sin(state['angle_rad'])
    
    rot_x = shifted_x * cos_a + shifted_y * sin_a
    rot_y = -shifted_x * sin_a + shifted_y * cos_a
    
    # 4. Shift back to rotated pixel space and subtract crop offset
    final_px_x = (rot_x + state['center_rot_px'][0]) - state['x_crop_offset']
    final_px_y = (rot_y + state['center_rot_px'][1]) - state['y_crop_offset']
    
    # 5. Convert to Local Meters (Origin 0,0)
    # Total rows in the final rectified DEM
    new_nrows = (state['center_rot_px'][1] * 2) - 2 * state['y_crop_offset']
    local_x = final_px_x * state['cellsize']
    local_y = (new_nrows - final_px_y) * state['cellsize']
    
    # Create DataFrame without column names for raw output
    transformed_data = np.column_stack((local_x, local_y))
    
    # Save to CSV: index=False (no row numbers), header=False (no column names)
    np.savetxt(csv_output, transformed_data, delimiter=',', fmt='%.4f')
    
    return transformed_data
# EXECUTION
new_points = transform_csv_coordinates('mesh/dam_break_finer_area_UTM14.csv', 'mesh/dam_break_finer_area_rectified.csv', state)